# QC Cross-Attention Source Importance
Loads `qc_attn_weights.npz` produced by `state tx predict --save-attn-weights` and
visualises which embedding sources the model attends to most.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Point this at your qc_attn_weights.npz ──────────────────────────────────
NPZ_PATH = Path("/dcai/users/hilarn/55_cu_0055/code/enhance_state/results/30"
                "/qc_emb_30_lr1e-4/qc_emb_30_lr1e-4"
                "/eval_last.ckpt/qc_attn_weights.npz")
# ─────────────────────────────────────────────────────────────────────────────

d = np.load(NPZ_PATH, allow_pickle=True)
weights      = d["attn_weights"]   # (N_cells, N_sources)  float32
pert_names   = d["pert_names"]     # (N_cells,)
celltypes    = d["celltypes"]      # (N_cells,)
source_names = d["source_names"]   # (N_sources,)

print(f"Cells: {weights.shape[0]:,}   Sources: {weights.shape[1]}")
print("Sources:", list(source_names))

## 1 · Overall mean attention per source

In [ ]:
mean_w = weights.mean(axis=0)          # (N_sources,)
order  = np.argsort(mean_w)[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(source_names)), mean_w[order], color="steelblue")
ax.set_xticks(range(len(source_names)))
ax.set_xticklabels(source_names[order], rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Mean attention weight")
ax.set_title("Overall source importance (mean across all cells)")
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "source_importance_overall.pdf", dpi=150)
plt.show()

## 2 · Per-source distribution (violin)

In [ ]:
df = pd.DataFrame(weights, columns=source_names)
df_long = df.melt(var_name="source", value_name="attn_weight")

fig, ax = plt.subplots(figsize=(10, 4))
sns.violinplot(data=df_long, x="source", y="attn_weight",
               order=source_names[order], cut=0, ax=ax)
ax.set_xticklabels(source_names[order], rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Attention weight")
ax.set_title("Per-source attention distribution across all cells")
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "source_importance_violin.pdf", dpi=150)
plt.show()

## 3 · Heatmap — mean attention per perturbation

In [ ]:
df["pert"] = pert_names
pert_mean = df.groupby("pert")[list(source_names)].mean()  # (N_perts, N_sources)

# Sort perts by their dominant source
pert_order = pert_mean.values.argmax(axis=1).argsort()
pert_mean_sorted = pert_mean.iloc[pert_order]

fig, ax = plt.subplots(figsize=(max(6, len(source_names) * 0.8),
                                min(40, len(pert_mean) * 0.18 + 2)))
sns.heatmap(pert_mean_sorted[source_names[order]],
            ax=ax, cmap="YlOrRd", linewidths=0,
            xticklabels=True, yticklabels=(len(pert_mean) < 80))
ax.set_title("Mean attention per perturbation × source")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "source_importance_per_pert.pdf", dpi=150)
plt.show()

print(f"\n{len(pert_mean):,} perturbations")

## 4 · Top-source per perturbation

In [ ]:
top_source = pert_mean[list(source_names)].idxmax(axis=1)
counts = top_source.value_counts()

fig, ax = plt.subplots(figsize=(7, 3))
counts.plot.bar(ax=ax, color="steelblue")
ax.set_ylabel("Number of perturbations")
ax.set_title("Dominant source per perturbation")
ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "dominant_source_per_pert.pdf", dpi=150)
plt.show()

print(counts.to_string())

## 5 · Source correlation (do sources co-attend?)

In [ ]:
corr = pd.DataFrame(weights, columns=source_names).corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title("Cross-source attention correlation")
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "source_correlation.pdf", dpi=150)
plt.show()